# Problema canibalilor și misionarilor

Pe malul râului se află N canibali și N misionari. Lângă ei se află o barcă cu maxim. M locuri. Toată lumea vrea să ajungă pe partea cealaltă a râului, dar nimeni nu știe să înoate.

Nici pe barcă, nici pe vreun mal nu trebuie să fie vreodată mai mulți canibali decât misionari (altfel o să îi mănânce). În barcă trebuie să fie mereu minim un om și maxim M oameni. Permutările între mal și barcă au loc instantaneu, așadar nu ne punem problema ordinii în care coboară sau urcă oamenii în barcă.

Care este o secvență de acțiuni în urma căreia toți oamenii să ajungă pe celălalt mal fără ca misionarii să ajungă prânz pentru canibali?

*Problema clasica este cu N=3 (canibali / misionari) și M=2 (locuri în barcă).*

## 1. Clasa Nod

Modificați informația clasei Nod pentru datele jocului curent. Noua informație va include:
* numărul de canibali pe malul stâng;
* numărul de misionari pe malul stâng;
* poziția bărcii.

Citiți dintr-un **_fișier_** valorile parametrilor N și M și păstrați-le ca elemente statice în clasă. Puteți pune valori default în clasă N=3 și M=2.

Implementați următoarele metode:
* constructorul clasei -> *\_\_init__*
* egalitatea -> *\_\_eq__*
* transformarea în obiect de tip string a informației nodului curent -> *\_\_str__*
* transformarea în obiect de tip string a informației elementelor dintr-o listă -> *\_\_repr__*

_Observații:_
* Informația nodului curent va fi afișată de forma:
```
Stare curentă:
3 misionari, 3 canibali  | 0 misionari, 0 canibali
Barca se află pe malul stâng
```
* Informația elementului dintr-o listă va fi afișată de forma:
```
(3, 3, 0)
```
Unde tuplul reprezintă (<număr misionari pe malul stâng>, <număr canibali pe malul stâng>, <malul pe care se află barca>)

In [16]:
# citim datele de la tastatura
# class nod cu init eq str repr
class Nod:
    N = 3 # misionari canibali ceva
    M = 2 # locuri in barca
    def __init__(self, canibali : str, misionari : str, pozitie : int, parinte = None):
        self.canibali = canibali
        self.misionari = misionari
        self.pozitie = pozitie
        self.parinte = parinte

    def __eq__(self, other):
        return self.canibali == other.canibali and self.misionari == other.misionari and self.pozitie == other.pozitie

    def __str__(self):
        ms = self.misionari
        cs = self.canibali
        md = Nod.N - ms
        cd = Nod.N - cs

        mal_barca = "stang" if self.pozitie == 0 else "drept"

        rezultat =  f"Stare curenta:\n"
        rezultat += f"{ms} misionari, {cs} canibali  |  {md} misionari, {cd} canibali\n"
        rezultat += f"Barca se afla pe malul {mal_barca}"
        return rezultat

    def __repr__(self):
        return f"({self.misionari}, {self.canibali}, {self.pozitie})"

    def conditie(self):
        if self.misionari > 0 and self.canibali > self.misionari:
            return False
        ms_dreapta = Nod.N - self.misionari
        cs_dreapta = Nod.N - self.canibali
        if ms_dreapta > 0 and cs_dreapta > ms_dreapta:
            return False
        return True

    def drumRadacina (self):
        noduri = []
        temp = self
        while temp is not None:
            noduri.append(temp)
            temp = temp.parinte
        noduri.reverse()
        return noduri

    def vizitat (self, nod) -> bool:
       for nod_vechi in self.drumRadacina():
           if nod == nod_vechi:
               return True
       return False

n = Nod(3, 3, 0, None)
print(n)
print([n])

Stare curenta:
3 misionari, 3 canibali  |  0 misionari, 0 canibali
Barca se afla pe malul stang
[(3, 3, 0)]


## 2. Clasa Graf

Modificați următoarele funcții din clasa Graf pentru problema curentă:
* scop - verifică dacă toți oamenii au ajuns pe malul drept
* succesori - generează lista de succesori valizi ai stării curente

_Sugestie:_ Pentru a scrie mai puțin, puteți calcula succesorii unei stări în clasa Nod și apela acea funcție din clasa Graf. Nu uitați să verificați că succesorul nu a mai fost vizitat în drumul de la rădăcină!

In [25]:
class Graf:
    def __init__(self, nodStart):
        self.nodStart = nodStart

    def scop(self, nod) -> bool:
        return nod.canibali == 0 and nod.misionari == 0 and nod.pozitie == 1

    def succesori(self, nod) -> list[Nod]:
        lista_succesori = []
        poz2 = (1 + nod.pozitie) % 2
        for i in range(Nod.N + 1):
            for j in range(Nod.N + 1):
                if 1 <= i+j <= Nod.M:
                    if nod.pozitie == 0:
                        m2 = nod.misionari - i
                        c2 = nod.canibali - j
                    else:
                        m2 = nod.misionari + i
                        c2 = nod.canibali + j

                    if 0 <= m2 <= Nod.N and 0 <= c2 <= Nod.N:
                        nou_nod = Nod(c2, m2, poz2, nod)
                        if nou_nod.conditie() and not nod.vizitat(nou_nod):
                            lista_succesori.append(nou_nod)
        return lista_succesori
# ------ test:
stare_start = Nod(canibali=3, misionari=3, pozitie=0)
graf = Graf(stare_start)
print(stare_start)

succesori_start = graf.succesori(stare_start)

print(f"Succesori: ")
for s in succesori_start:
    print(repr(s))


Stare curenta:
3 misionari, 3 canibali  |  0 misionari, 0 canibali
Barca se afla pe malul stang
Succesori: 
(3, 2, 1)
(3, 1, 1)
(2, 2, 1)


## 3. Căutarea drumului

Creați o funcție *printDrumRadacina()* care returnează drumul până la nodul curent în formatul de mai jos. Rulați problema cu BFS / DFS pentru N=3, M=2 și NSOL=2 (numărul de soluții cerute) și afișați într-un fișier cu calea dată de la tastatură drumul de la rădăcină până la nodul curent (în formatul precizat anterior) și timpul de rulare al algoritmului:

```
Stare curentă:
3 misionari, 3 canibali  | 0 misionari, 0 canibali
Barca se află pe malul stâng

Barca s-a deplasat de pe malul stâng pe malul drept cu 0 misionari și 2 canibali.

Stare curentă:
3 misionari, 1 canibali  | 0 misionari, 2 canibali
Barca se află pe malul drept

...

Stare curentă:
0 misionari, 0 canibali  | 3 misionari, 3 canibali
Barca se află pe malul drept

Timpul de rulare: 0.00021839141845703125 secunde.
```

In [36]:
# bfs i guess
def printDrum(nod):
    drum = nod.drumRadacina()
    for n in drum:
        print("ne plimbam de la un mal la altul") # mai am de scris cu cati canibali si cati misionari
        print(n)
    print(len(drum))
    return True

def bfs(graf, n):
    q = [graf.nodStart]
    solutii = 0
    while q and solutii < n:
        current = q.pop(0)
        if graf.scop(current):
            print(f"Sol:")
            printDrum(current)
            solutii += 1
            if solutii == n:
                return
        lista_succesori = graf.succesori(current)
        q.extend(lista_succesori)

n = int(input("nr sol: "))
bfs(graf, n)

Sol:
ne plimbam de la un mal la altul
Stare curenta:
3 misionari, 3 canibali  |  0 misionari, 0 canibali
Barca se afla pe malul stang
ne plimbam de la un mal la altul
Stare curenta:
3 misionari, 1 canibali  |  0 misionari, 2 canibali
Barca se afla pe malul drept
ne plimbam de la un mal la altul
Stare curenta:
3 misionari, 2 canibali  |  0 misionari, 1 canibali
Barca se afla pe malul stang
ne plimbam de la un mal la altul
Stare curenta:
3 misionari, 0 canibali  |  0 misionari, 3 canibali
Barca se afla pe malul drept
ne plimbam de la un mal la altul
Stare curenta:
3 misionari, 1 canibali  |  0 misionari, 2 canibali
Barca se afla pe malul stang
ne plimbam de la un mal la altul
Stare curenta:
1 misionari, 1 canibali  |  2 misionari, 2 canibali
Barca se afla pe malul drept
ne plimbam de la un mal la altul
Stare curenta:
2 misionari, 2 canibali  |  1 misionari, 1 canibali
Barca se afla pe malul stang
ne plimbam de la un mal la altul
Stare curenta:
0 misionari, 2 canibali  |  3 misionari, 1 c

## 4. Problemă alternativă

Modificați problema astfel încât în starea inițială avem un număr NM de misionari pe malul stâng și un alt număr NC de canibali pe malul drept, iar toată lumea își dorește să traverseze râul.

Canibalii pot mânca misionarii doar pe malul drept sau în barcă (malul stâng este foarte aproape de tabăra misionarilor în care au fost plasate mai mult de NC gărzi care se asigură că totul este în regulă).

Barca se află inițial pe malul stâng (al misionarilor) întrucât aceștia sunt singurii care știu să o conducă.

NM > 0, NC > 0, M >= 2

**Atenție:** asta înseamnă că în barcă trebuie să fie mereu minim un misionar.